In [56]:
# Standard library imports
import os
import sys
import random
import warnings
import math

# Third-party numerical and data handling
import numpy as np
import pandas as pd
import h5py
import cv2
from PIL import Image

# Visualization
import matplotlib.pyplot as plt
from tqdm import tqdm

# Machine learning utilities
from sklearn.metrics import (
    average_precision_score,
    label_ranking_average_precision_score,
    roc_auc_score
)

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

# PyTorch vision
from torchvision import models

# Albumentations
import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from albumentations.pytorch import ToTensorV2

# Custom modules (Kaggle inputs)
sys.path.append("/kaggle/input/asymmetric-loss-dataset")
sys.path.append("/kaggle/input/data-preprocessing")
sys.path.append("/kaggle/input/data-transformations")

from losses import AsymmetricLossOptimized
from preprocess import ecg_processing_pipeline, smart_pad_and_resize_ecg
from transformations import CornerCutout, GradientShadow, PaperFoldEffect, BottomBlur

In [57]:
def check_device():
    """
    Check available compute devices and return the best one.
    Priority: CUDA > MPS > CPU
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("✓ CUDA available")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("✓ MPS (Apple Silicon GPU) available")
    else:
        device = torch.device("cpu")
        print("✗ Using CPU (no GPU acceleration available)")
    
    print(f"\nSelected device: {device}")
    return device

# Check and get device
device = check_device()

✓ MPS (Apple Silicon GPU) available

Selected device: mps


In [ ]:
class Head(nn.Module):
    def __init__(self, in_features, hidden_layer, dropout_rate=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_layer),
            nn.BatchNorm1d(hidden_layer),  
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer, hidden_layer // 2), 
            nn.BatchNorm1d(hidden_layer // 2),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer // 2, 1)
        )
    
    def forward(self, x):
        return self.layers(x)

class MultiHeadEfficientNet(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3):
        super().__init__()
        
        backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
        in_features = backbone.classifier[2].in_features
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"
        
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features), 
            nn.BatchNorm1d(in_features),
            nn.GELU(), 
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate)
        )
        
        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate) 
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        processed_feats = self.shared_feature_processor(backbone_feats)
        outputs = [head(processed_feats) for head in self.heads]
        return torch.cat(outputs, dim=1)  # [batch, num_conditions]

In [ ]:
backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)


1024